<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_20_End_to_End_RAG_Product.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q langchain langchain-community faiss-cpu fastapi uvicorn pyngrok nest-asyncio pandas sentence-transformers transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
!pip install -q "requests==2.32.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [1]:
!pip install -q --force-reinstall "requests>=2.32.5"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.0/137.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.6/250.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
raw_docs = [
    {
        "id": "doc_001",
        "title": "Password Reset Policy",
        "category": "account",
        "content": """Users can reset their password from the login page by clicking
        'Forgot Password'. A reset link is sent to the registered email and expires
        after 30 minutes. If the link expires, the user must request a new one.
        Passwords must be at least 10 characters and include one number and one
        special character. Support agents cannot reset a password manually for
        security reasons; they can only trigger a new reset email."""
    },
    {
        "id": "doc_002",
        "title": "Subscription Billing Cycle",
        "category": "billing",
        "content": """Subscriptions renew automatically every 30 days from the
        original signup date. Invoices are generated 3 days before renewal and
        sent via email. Failed payments trigger 3 retry attempts over 7 days
        before the subscription is downgraded to the free tier. Refunds are only
        issued within 14 days of the original charge and must be requested through
        support, not self-service."""
    },
    {
        "id": "doc_003",
        "title": "API Rate Limits",
        "category": "technical",
        "content": """The public API allows 100 requests per minute per API key on
        the free tier and 1000 requests per minute on the paid tier. Exceeding the
        limit returns a 429 status code with a Retry-After header. Rate limits reset
        on a rolling 60-second window, not a fixed clock minute. Enterprise
        customers can request custom limits by contacting sales."""
    },
    {
        "id": "doc_004",
        "title": "Data Export Process",
        "category": "technical",
        "content": """Users can export their account data as a CSV or JSON file from
        Settings > Data > Export. Exports are processed asynchronously and a
        download link is emailed within 24 hours. Exported data includes account
        metadata, activity logs from the last 12 months, and uploaded files under
        500MB. Larger files must be requested via support."""
    },
    {
        "id": "doc_005",
        "title": "Team Permissions Overview",
        "category": "account",
        "content": """Workspaces support three roles: Admin, Editor, and Viewer.
        Admins can manage billing, invite or remove members, and change roles.
        Editors can create and edit content but cannot manage billing or members.
        Viewers have read-only access. Role changes take effect immediately and do
        not require the affected user to log out."""
    },
    {
        "id": "doc_006",
        "title": "Cancellation Policy",
        "category": "billing",
        "content": """Users can cancel a subscription at any time from Settings >
        Billing > Cancel Plan. Cancellation takes effect at the end of the current
        billing cycle; there are no partial-month refunds for early cancellation.
        Cancelled accounts retain access to paid features until the cycle ends,
        then automatically revert to the free tier. Data is retained for 90 days
        after downgrade before deletion."""
    },
]

print(f"Loaded {len(raw_docs)} source documents")

Loaded 6 source documents


In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def build_chunks(raw_docs, chunk_size=300, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for doc in raw_docs:
        pieces = splitter.split_text(doc["content"].strip())
        for i, piece in enumerate(pieces):
            chunk_id = f"{doc['id']}_chunk{i}"
            chunks.append(
                Document(
                    page_content=piece,
                    metadata={
                        "chunk_id": chunk_id,
                        "source_id": doc["id"],
                        "title": doc["title"],
                        "category": doc["category"],
                    },
                )
            )
    return chunks

chunks = build_chunks(raw_docs)
print(f"Created {len(chunks)} chunks from {len(raw_docs)} documents")

ModuleNotFoundError: No module named 'langchain.text_splitter'

In [4]:
!pip install -q langchain-text-splitters


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def build_chunks(raw_docs, chunk_size=300, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for doc in raw_docs:
        pieces = splitter.split_text(doc["content"].strip())
        for i, piece in enumerate(pieces):
            chunk_id = f"{doc['id']}_chunk{i}"
            chunks.append(
                Document(
                    page_content=piece,
                    metadata={
                        "chunk_id": chunk_id,
                        "source_id": doc["id"],
                        "title": doc["title"],
                        "category": doc["category"],
                    },
                )
            )
    return chunks

chunks = build_chunks(raw_docs)
print(f"Created {len(chunks)} chunks from {len(raw_docs)} documents")

Created 12 chunks from 6 documents


In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# all-MiniLM-L6-v2 runs locally on CPU, no API key, ~80MB download
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

print("FAISS index built and saved to ./faiss_index")

/tmp/ipykernel_5146/2579125113.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_5146/2579125113.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS index built and saved to ./faiss_index


In [7]:
def retrieve(query, k=4, category=None):
    filter_dict = {"category": category} if category else None
    results = vectorstore.similarity_search_with_relevance_scores(
        query, k=k, filter=filter_dict
    )
    docs = [r[0] for r in results]
    scores = [r[1] for r in results]
    return docs, scores

test_docs, test_scores = retrieve("how do I reset my password")
for d, s in zip(test_docs, test_scores):
    print(f"{s:.3f}  {d.metadata['title']}  ({d.metadata['chunk_id']})")

0.473  Password Reset Policy  (doc_001_chunk0)
0.274  Password Reset Policy  (doc_001_chunk1)
-0.170  Cancellation Policy  (doc_006_chunk1)
-0.224  Data Export Process  (doc_004_chunk0)


/tmp/ipykernel_5146/491277664.py:3: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='e562b208-35a3-4655-a035-819c496850b7', metadata={'chunk_id': 'doc_001_chunk0', 'source_id': 'doc_001', 'title': 'Password Reset Policy', 'category': 'account'}, page_content="Users can reset their password from the login page by clicking\n        'Forgot Password'. A reset link is sent to the registered email and expires\n        after 30 minutes. If the link expires, the user must request a new one."), np.float32(0.47260356)), (Document(id='742dd7d1-56fd-4b54-8027-7a7283112f0c', metadata={'chunk_id': 'doc_001_chunk1', 'source_id': 'doc_001', 'title': 'Password Reset Policy', 'category': 'account'}, page_content='Passwords must be at least 10 characters and include one number and one\n        special character. Support agents cannot reset a password manually for\n        security reasons; they can only trigger a new reset email.'), np.float32(0.27377635)), (Document(id='73383f1

In [8]:
from transformers import pipeline

GROUNDING_SYSTEM_PROMPT = """Answer ONLY using the given context. If the context
does not contain the answer, say "I don't have enough information in the
knowledge base to answer that." Do not use outside knowledge. Be concise."""

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=200,
)

def build_context(docs):
    blocks = []
    for d in docs:
        blocks.append(f"[{d.metadata['chunk_id']}] ({d.metadata['title']}): {d.page_content}")
    return "\n\n".join(blocks)

def generate_answer(query, docs):
    context = build_context(docs)
    prompt = f"{GROUNDING_SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    result = generator(prompt, do_sample=False)[0]["generated_text"]
    return result.strip()

# quick test
docs, scores = retrieve("How do I reset my password?")
print(generate_answer("How do I reset my password?", docs))

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

KeyError: "Unknown task text2text-generation, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [9]:
!pip install -q sentencepiece protobuf

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GROUNDING_SYSTEM_PROMPT = """Answer ONLY using the given context. If the context
does not contain the answer, say "I don't have enough information in the
knowledge base to answer that." Do not use outside knowledge. Be concise."""

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def build_context(docs):
    blocks = []
    for d in docs:
        blocks.append(f"[{d.metadata['chunk_id']}] ({d.metadata['title']}): {d.page_content}")
    return "\n\n".join(blocks)

def generate_answer(query, docs):
    context = build_context(docs)
    prompt = f"{GROUNDING_SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

# quick test
docs, scores = retrieve("How do I reset my password?")
print(generate_answer("How do I reset my password?", docs))

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/tmp/ipykernel_5146/491277664.py:3: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='e562b208-35a3-4655-a035-819c496850b7', metadata={'chunk_id': 'doc_001_chunk0', 'source_id': 'doc_001', 'title': 'Password Reset Policy', 'category': 'account'}, page_content="Users can reset their password from the login page by clicking\n        'Forgot Password'. A reset link is sent to the registered email and expires\n        after 30 minutes. If the link expires, the user must request a new one."), np.float32(0.4663086)), (Document(id='742dd7d1-56fd-4b54-8027-7a7283112f0c', metadata={'chunk_id': 'doc_001_chunk1', 'source_id': 'doc_001', 'title': 'Password Reset Policy', 'category': 'account'}, page_content='Passwords must be at least 10 characters and include one number and one\n        special character. Support agents cannot reset a password manually for\n        security reasons; they can only trigger a new reset email.'), np.float32(0.28169966)), (Document(id='73383f1f

Click 'Forgot Password'.


In [11]:
%%writefile rag_pipeline.py
"""
End-to-end RAG pipeline — fully local, no external API calls.
"""

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

GROUNDING_SYSTEM_PROMPT = """Answer ONLY using the given context. If the context
does not contain the answer, say "I don't have enough information in the
knowledge base to answer that." Do not use outside knowledge. Be concise."""

LOW_CONFIDENCE_THRESHOLD = 0.3

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(
    "faiss_index", embeddings, allow_dangerous_deserialization=True
)

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


def retrieve(query: str, k: int = 4, category: str = None):
    filter_dict = {"category": category} if category else None
    results = vectorstore.similarity_search_with_relevance_scores(
        query, k=k, filter=filter_dict
    )
    docs = [r[0] for r in results]
    scores = [r[1] for r in results]
    return docs, scores


def build_context(docs):
    blocks = []
    for d in docs:
        blocks.append(f"[{d.metadata['chunk_id']}] ({d.metadata['title']}): {d.page_content}")
    return "\n\n".join(blocks)


def generate_answer(query: str, docs):
    context = build_context(docs)
    prompt = f"{GROUNDING_SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


def ask(query: str, k: int = 4, category: str = None):
    docs, scores = retrieve(query, k=k, category=category)

    if not docs:
        return {
            "answer": "I don't have enough information in the knowledge base to answer that.",
            "sources": [],
            "top_score": 0.0,
        }

    answer = generate_answer(query, docs)
    top_score = max(scores)

    if top_score < LOW_CONFIDENCE_THRESHOLD:
        answer += (
            "\n\n⚠️ Low confidence: the retrieved sources are only weakly related "
            "to this question. This answer may not be well-supported."
        )

    sources = [
        {
            "chunk_id": d.metadata["chunk_id"],
            "title": d.metadata["title"],
            "score": round(s, 3),
        }
        for d, s in zip(docs, scores)
    ]

    return {"answer": answer, "sources": sources, "top_score": round(top_score, 3)}

Writing rag_pipeline.py


In [12]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
from rag_pipeline import ask

app = FastAPI(title="Local RAG Knowledge Assistant")


class AskRequest(BaseModel):
    query: str
    category: str | None = None
    k: int | None = 4


class SourceItem(BaseModel):
    chunk_id: str
    title: str
    score: float


class AskResponse(BaseModel):
    answer: str
    sources: list[SourceItem]
    top_score: float


@app.post("/ask", response_model=AskResponse)
def ask_endpoint(req: AskRequest):
    result = ask(req.query, k=req.k or 4, category=req.category)
    return result


@app.get("/health")
def health():
    return {"status": "ok"}

Writing app.py


In [13]:
import nest_asyncio, uvicorn, threading, time

nest_asyncio.apply()

def run_server():
    uvicorn.run("app:app", host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)
print("Server running at http://localhost:8000")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Server running at http://localhost:8000


In [14]:
import requests

resp = requests.post(
    "http://localhost:8000/ask",
    json={"query": "How do I reset my password?"},
)
print(resp.status_code)
print(resp.json())

/content/rag_pipeline.py:27: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='e562b208-35a3-4655-a035-819c496850b7', metadata={'chunk_id': 'doc_001_chunk0', 'source_id': 'doc_001', 'title': 'Password Reset Policy', 'category': 'account'}, page_content="Users can reset their password from the login page by clicking\n        'Forgot Password'. A reset link is sent to the registered email and expires\n        after 30 minutes. If the link expires, the user must request a new one."), np.float32(0.4663086)), (Document(id='742dd7d1-56fd-4b54-8027-7a7283112f0c', metadata={'chunk_id': 'doc_001_chunk1', 'source_id': 'doc_001', 'title': 'Password Reset Policy', 'category': 'account'}, page_content='Passwords must be at least 10 characters and include one number and one\n        special character. Support agents cannot reset a password manually for\n        security reasons; they can only trigger a new reset email.'), np.float32(0.28169966)), (Document(id='73383f1f-a292-4

200
{'answer': "Click 'Forgot Password'.", 'sources': [{'chunk_id': 'doc_001_chunk0', 'title': 'Password Reset Policy', 'score': 0.4659999907016754}, {'chunk_id': 'doc_001_chunk1', 'title': 'Password Reset Policy', 'score': 0.28200000524520874}, {'chunk_id': 'doc_006_chunk1', 'title': 'Cancellation Policy', 'score': -0.164000004529953}, {'chunk_id': 'doc_004_chunk0', 'title': 'Data Export Process', 'score': -0.23000000417232513}], 'top_score': 0.4659999907016754}


In [15]:
import pandas as pd

test_queries = [
    {"query": "How do I reset my password?", "difficulty": "easy"},
    {"query": "What roles exist for team workspaces?", "difficulty": "easy"},
    {"query": "How often does a subscription renew?", "difficulty": "easy"},
    {"query": "What format can I export my data in?", "difficulty": "easy"},
    {"query": "What is the API rate limit on the free tier?", "difficulty": "easy"},
    {"query": "What happens if a payment fails?", "difficulty": "medium"},
    {"query": "Can support agents reset a password for me directly?", "difficulty": "medium"},
    {"query": "What happens to my data after I downgrade my account?", "difficulty": "medium"},
    {"query": "Can an Editor change another member's role?", "difficulty": "medium"},
    {"query": "What status code do I get if I exceed the rate limit?", "difficulty": "medium"},
    {"query": "If I cancel today, will I get a refund for the unused days this month?", "difficulty": "hard"},
    {"query": "How long is a password reset link valid, and what happens if it expires?", "difficulty": "hard"},
    {"query": "Can I get a custom API rate limit, and who do I contact?", "difficulty": "hard"},
    {"query": "What's the maximum file size for a self-service data export?", "difficulty": "hard"},
    {"query": "What is your refund policy for annual enterprise contracts?", "difficulty": "hard"},
]

from rag_pipeline import ask as ask_fn  # reuse the module directly, no server round trip

results = []
for item in test_queries:
    r = ask_fn(item["query"])
    results.append({
        "query": item["query"],
        "difficulty": item["difficulty"],
        "answer": r["answer"],
        "top_score": r["top_score"],
        "sources": ", ".join(s["chunk_id"] for s in r["sources"]),
        "retrieval_quality": "good" if r["top_score"] >= 0.5 else ("weak" if r["top_score"] >= 0.3 else "poor"),
    })

df = pd.DataFrame(results)
df.to_csv("test_results.csv", index=False)
df

/content/rag_pipeline.py:27: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='3d2b0ee7-59fc-4470-b2c0-7f6e857fbe4e', metadata={'chunk_id': 'doc_005_chunk0', 'source_id': 'doc_005', 'title': 'Team Permissions Overview', 'category': 'account'}, page_content='Workspaces support three roles: Admin, Editor, and Viewer.\n        Admins can manage billing, invite or remove members, and change roles.\n        Editors can create and edit content but cannot manage billing or members.'), np.float32(0.49673474)), (Document(id='6b9fb3d9-8286-4326-b0e0-30d11bd3286c', metadata={'chunk_id': 'doc_005_chunk1', 'source_id': 'doc_005', 'title': 'Team Permissions Overview', 'category': 'account'}, page_content='Viewers have read-only access. Role changes take effect immediately and do\n        not require the affected user to log out.'), np.float32(-0.10171807)), (Document(id='2369de95-69f7-4f63-a6ab-0384a06406cc', metadata={'chunk_id': 'doc_004_chunk1', 'source_id': 'doc_004', 'ti

,query,difficulty,answer,top_score,sources,retrieval_quality
0,How do I reset my password?,easy,Click 'Forgot Password'.,0.466,"doc_001_chunk0, doc_001_chunk1, doc_006_chunk1...",weak
1,What roles exist for team workspaces?,easy,"Admin, Editor, and Viewer.",0.497,"doc_005_chunk0, doc_005_chunk1, doc_004_chunk1...",weak
2,How often does a subscription renew?,easy,every 30 days.,0.617,"doc_002_chunk0, doc_006_chunk1, doc_006_chunk0...",good
3,What format can I export my data in?,easy,CSV or JSON.\n\n⚠️ Low confidence: the retriev...,0.241,"doc_004_chunk0, doc_004_chunk1, doc_001_chunk1...",poor
4,What is the API rate limit on the free tier?,easy,100 requests per minute per API key.,0.691,"doc_003_chunk0, doc_002_chunk0, doc_006_chunk1...",good
5,What happens if a payment fails?,medium,3 retry attempts over 7 days before the subscr...,0.073,"doc_002_chunk0, doc_002_chunk1, doc_006_chunk1...",poor
6,Can support agents reset a password for me dir...,medium,I don't have enough information in the knowled...,0.641,"doc_001_chunk1, doc_001_chunk0, doc_005_chunk1...",good
7,What happens to my data after I downgrade my a...,medium,Data is retained for 90 days after downgrade b...,0.443,"doc_006_chunk1, doc_004_chunk0, doc_001_chunk0...",weak
8,Can an Editor change another member's role?,medium,No.,0.322,"doc_005_chunk0, doc_005_chunk1, doc_001_chunk1...",weak
9,What status code do I get if I exceed the rate...,medium,429.,0.443,"doc_003_chunk0, doc_003_chunk1, doc_002_chunk1...",weak


In [16]:
%%writefile README.md
# Local RAG Knowledge Assistant (No External API)

A FastAPI service answering questions over a small knowledge base using
retrieval-augmented generation. Fully local: embeddings via
`sentence-transformers` and generation via a local HuggingFace model
(`google/flan-t5-base`). No OpenAI key or paid API required.

## Setup

```bash
pip install langchain langchain-community faiss-cpu fastapi uvicorn \
    sentence-transformers transformers accelerate
```

No environment variables are required — everything runs on-device. First run
will download the embedding model (~80MB) and the generation model (~250MB)
from HuggingFace.

## Build the index (one-time)

Run the chunking + embedding steps (see `rag_pipeline.py` for the FAISS
build logic, or run the notebook cells that create `faiss_index/`).

## Running the API locally

```bash
uvicorn app:app --host 0.0.0.0 --port 8000
```

## Example request

```bash
curl -X POST "http://localhost:8000/ask" \
  -H "Content-Type: application/json" \
  -d '{"query": "How do I reset my password?"}'
```

Example response:
```json
{
  "answer": "You can reset your password from the login page by clicking Forgot Password...",
  "sources": [
    {"chunk_id": "doc_001_chunk0", "title": "Password Reset Policy", "score": 0.42}
  ],
  "top_score": 0.42
}
```

## Confidence indicator

If the top retrieval score for a query falls below the configured threshold
(default 0.3, tune in `rag_pipeline.py` based on your embedding model's score
distribution), a low-confidence warning is appended to the answer.

## Endpoints

- `POST /ask` — body: `{"query": str, "category": str (optional), "k": int (optional)}`
- `GET /health` — health check

## Notes

- Swap `google/flan-t5-base` for a larger model (`flan-t5-large`, or a
  Qwen/Llama instruct model) if you have GPU access and want higher-quality
  generation.
- Swap the sample `raw_docs` for your own dataset — the pipeline is
  content-agnostic.

Writing README.md


In [17]:
from google.colab import files

files.download("rag_pipeline.py")
files.download("app.py")
files.download("README.md")
files.download("test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
%%writefile README.md
# Local RAG Knowledge Assistant (No External API)

A FastAPI service that answers questions over a small internal knowledge base
using retrieval-augmented generation (RAG) — fully local, no OpenAI key or
paid API required. Embeddings run via `sentence-transformers` and answer
generation runs via a local HuggingFace seq2seq model (`google/flan-t5-base`).

## Architecture

1. **Chunking** — source docs are split into overlapping chunks with
   `RecursiveCharacterTextSplitter`, preserving title/category/chunk-id
   metadata for citation and filtering.
2. **Semantic search** — chunks are embedded with
   `sentence-transformers/all-MiniLM-L6-v2` and indexed in FAISS.
3. **Metadata filtering** — retrieval can be scoped by `category`
   (e.g. `billing`, `technical`, `account`).
4. **Grounded generation** — a system prompt forces the model to answer only
   from retrieved context, or say it doesn't know.
5. **Confidence indicator** — if the top similarity score for a query falls
   below a threshold (default `0.3`), a low-confidence warning is appended
   to the answer.

## Setup

```bash
pip install langchain langchain-community langchain-text-splitters \
    faiss-cpu fastapi uvicorn sentence-transformers transformers \
    accelerate sentencepiece protobuf
```

No environment variables are required — everything runs on-device. First run
downloads the embedding model (~80MB) and generation model (~250MB) from
HuggingFace automatically.

## Build the index (one-time)

Run the chunking + embedding step to produce a local `faiss_index/` folder
(see the chunking and FAISS-build cells / script). `rag_pipeline.py` loads
this index at import time.

## Running the API locally

```bash
uvicorn app:app --host 0.0.0.0 --port 8000
```

## Example request

```bash
curl -X POST "http://localhost:8000/ask" \
  -H "Content-Type: application/json" \
  -d '{"query": "How do I reset my password?"}'
```

Example response:
```json
{
  "answer": "You can reset your password from the login page by clicking Forgot Password. The reset link expires after 30 minutes.",
  "sources": [
    {"chunk_id": "doc_001_chunk0", "title": "Password Reset Policy", "score": 0.42}
  ],
  "top_score": 0.42
}
```

## Optional:

Overwriting README.md
